[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/Job_Classification_Model_Flex_Analyzer.ipynb)

# 🤖 Job Classification Model Flexibility Analyzer

**A unified notebook for analyzing job classification quality using multiple LLM backends (Claude, Llama, Gemini)**

This notebook takes batch classification data and resource files to perform detailed analysis on potential role confusion and provide a comprehensive critique of the AI's assignment, independent of the original 'confusion score' logic.

## Key Features:
- **Model Flexibility**: Easily switch between `CLAUDE_API`, `LLAMA_LOCAL`, and `GEMINI_API`.
- **Contextual Analysis**: Uses Ground Truth, KSACs, and Role Groupings for deep reasoning.
- **Targeted Output**: Focuses on identifying and justifying viable alternative classifications.

## Required Inputs:
- `Evaluation Resources.zip` containing:
  - `Sample_JDs.json`
  - `Job_Classifications_Batch.json`
  - `Ground_Truth_Masterfile.json`
  - `MNPS_KSACs.json`
  - `MNPS_Role_Groups_by_KSAC_Similarity_FINAL.json`


## 1. Configuration and Model Selection

In [ ]:
# =========================================================================
# 🚨 STEP 1: CHOOSE YOUR MODEL MODE
# =========================================================================

# Select one of the following modes:
# 1. 'CLAUDE_API' (Requires Anthropic API Key)
# 2. 'LLAMA_LOCAL' (Requires Colab GPU/VRAM for Llama-3.1-70B-Instruct)
# 3. 'GEMINI_API' (Requires Gemini API Key)
MODEL_MODE = 'CLAUDE_API'  # <<< CHANGE THIS TO SWITCH MODELS >>>

# =========================================================================
# 🔑 STEP 2: API Keys & General Configuration
# =========================================================================

ANTHROPIC_API_KEY = ""
GEMINI_API_KEY = ""

# General Settings
ZIP_FILE_PATH = "/content/Evaluation Resources.zip" # Expected location after upload
CURRENT_RUN_PATH = "/content/analysis_results"
MAX_TOKENS = 2000
TEMPERATURE = 0.0 # Use 0.0 for deterministic, analytical reasoning
MAX_RETRIES = 3
RETRY_DELAY = 1.0

# LLAMA Specific Settings (only used if MODEL_MODE = 'LLAMA_LOCAL')
LLAMA_MODEL_NAME = "meta-llama/Meta-Llama-3.1-70B-Instruct"
LLAMA_USE_4BIT = True # Recommended to save VRAM

# CLAUDE Specific Settings (only used if MODEL_MODE = 'CLAUDE_API')
CLAUDE_MODEL_NAME = "claude-sonnet-4-5-20250929" # Sonnet 4.5 is cost-optimized
CLAUDE_FALLBACK_MODEL = "claude-sonnet-4-20250514"
ENABLE_PROMPT_CACHING = True # Highly recommended for cost savings

# GEMINI Specific Settings (only used if MODEL_MODE = 'GEMINI_API')
GEMINI_MODEL_NAME = "gemini-2.5-pro"

# =========================================================================
# 🔄 STEP 3: Dependency Installation (Runs automatically)
# =========================================================================

import os
import json
import zipfile
import tempfile
import time
from typing import Dict, List, Optional
import pandas as pd
from tqdm.auto import tqdm
from google.colab import files, userdata

print(f"Selected Model Mode: {MODEL_MODE}")

install_command = "pip install -q pandas numpy matplotlib seaborn plotly networkx"

if MODEL_MODE == 'CLAUDE_API':
    install_command += " anthropic>=0.34.0"
    try:
        ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
        print("✅ Anthropic API key loaded from Colab secrets.")
    except:
        print("⚠️ ANTHROPIC_API_KEY not found in secrets. Please set it manually.")

elif MODEL_MODE == 'LLAMA_LOCAL':
    install_command += " torch torchvision torchaudio transformers accelerate bitsandbytes chromadb sentence-transformers"
    print("⚠️ LLAMA_LOCAL mode requires a Colab GPU runtime (A100/V100 recommended).")

elif MODEL_MODE == 'GEMINI_API':
    install_command += " google-genai"
    try:
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
        print("✅ Gemini API key loaded from Colab secrets.")
    except:
        print("⚠️ GEMINI_API_KEY not found in secrets. Please set it manually.")

else:
    raise ValueError(f"Invalid MODEL_MODE: {MODEL_MODE}. Choose 'CLAUDE_API', 'LLAMA_LOCAL', or 'GEMINI_API'.")

print("📦 Installing dependencies...")
os.system(install_command)
print("✅ Dependencies installed.")


## 2. File Upload and Data Loading

In [ ]:
print("⬆️ Please upload 'Evaluation Resources.zip' when prompted.")
uploaded = files.upload()

if 'Evaluation Resources.zip' in uploaded:
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        # Extract to a temp directory to handle nested zips/folders
        zip_ref.extractall("/content/")
    print("✅ Files uploaded and extracted.")
else:
    print("❌ 'Evaluation Resources.zip' not found. Please upload the file.")

In [ ]:
# =========================================================================
# 🆕 Data Loading and Merging Logic (Modified from Claude Analyzer)
# =========================================================================

def load_all_json_datasets(zip_path: str) -> Dict[str, List[Dict]]:
    """Recursively searches /content/ for JSON files and loads them."""
    datasets = {}
    json_files = [
        'Sample_JDs.json',
        'Job_Classifications_Batch.json',
        'Ground_Truth_Masterfile.json',
        'MNPS_KSACs.json',
        'MNPS_Role_Groups_by_KSAC_Similarity_FINAL.json'
    ]

    print("🔍 Searching for JSON files...")
    for filename in json_files:
        found = False
        # Search in /content/ and all subdirectories
        for root, dirs, files in os.walk("/content/"):
            if filename.lower() in [f.lower() for f in files]:
                file_path = os.path.join(root, filename)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        datasets[filename.replace('.json', '')] = json.load(f)
                    print(f"  ✅ Loaded {filename}: {len(datasets[filename.replace('.json', '')])} records")
                    found = True
                    break
                except Exception as e:
                    print(f"  ⚠️ Error loading {filename}: {e}")
        if not found:
            print(f"  ❌ File not found: {filename}")

    return datasets

def create_analysis_dataframe(datasets: Dict) -> pd.DataFrame:
    """Merges JDs and Batch data into a single DataFrame."""
    df_jds = pd.DataFrame(datasets.get('Sample_JDs', []))
    df_batch = pd.DataFrame(datasets.get('Job_Classifications_Batch', []))

    # 1:1 merge on 'Job Title' / 'job_title_original'
    df_jds['Job Title'] = df_jds['Job Title'].astype(str).str.strip()
    df_batch['job_title_original'] = df_batch['job_title_original'].astype(str).str.strip()

    # Use an inner merge to ensure a clean 1:1 relationship for analysis
    df_analysis = pd.merge(
        df_batch, df_jds,
        left_on='job_title_original',
        right_on='Job Title',
        how='inner',
        suffixes=('_batch', '_jd')
    )

    # Combine relevant JD columns for easy reference in the prompt
    def create_jd_text(row):
        return f"Position Summary: {row.get('Position Summary', 'N/A')}\n"\
               f"Essential Functions: {row.get('Essential Functions', 'N/A')}\n"\
               f"Education: {row.get('Education', 'N/A')}\n"\
               f"Work Experience: {row.get('Work Experience', 'N/A')}\n"\
               f"KSACs: {row.get('Knowledge, Skills and Abilities', 'N/A')}"

    df_analysis['full_job_description'] = df_analysis.apply(create_jd_text, axis=1)

    print(f"\n✅ Merged Analysis DataFrame created: {len(df_analysis)} records")
    if len(df_analysis) != len(df_batch):
        print("⚠️ WARNING: Merge count mismatch. Check data keys.")

    return df_analysis

all_datasets = load_all_json_datasets(ZIP_FILE_PATH)
df_analysis = create_analysis_dataframe(all_datasets)


## 3. Model Initialization (Dynamic)

In [ ]:
# =========================================================================
# 🧱 Model Setup Helpers
# =========================================================================

import warnings
warnings.filterwarnings('ignore')

llm_client = None
llm_tokenizer = None
llm_model_name = None
system_instruction_cache = None

def setup_llama_local():
    """Setup Llama for LLAMA_LOCAL mode."""
    global llm_client, llm_tokenizer, llm_model_name
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    llm_model_name = LLAMA_MODEL_NAME
    print(f"Loading LLAMA model: {llm_model_name}...")

    if LLAMA_USE_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(
            llm_model_name,
            quantization_config=quantization_config,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
            use_cache=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            llm_model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True
        )

    tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    llm_client = model
    llm_tokenizer = tokenizer
    print(f"✅ LLAMA loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

def setup_claude_api():
    """Setup Anthropic client for CLAUDE_API mode."""
    global llm_client, llm_model_name, system_instruction_cache
    from anthropic import Anthropic
    llm_model_name = CLAUDE_MODEL_NAME
    if not ANTHROPIC_API_KEY:
        raise ValueError("Anthropic API Key is required for CLAUDE_API mode.")
    llm_client = Anthropic(api_key=ANTHROPIC_API_KEY)
    system_instruction_cache = create_claude_system_prompt_cache()
    print(f"✅ Claude client initialized. Using model: {llm_model_name}")

def setup_gemini_api():
    """Setup Gemini client for GEMINI_API mode."""
    global llm_client, llm_model_name, system_instruction_cache
    from google import genai
    llm_model_name = GEMINI_MODEL_NAME
    if not GEMINI_API_KEY:
        raise ValueError("Gemini API Key is required for GEMINI_API mode.")

    # Initialize client (it will pick up the key from the environment/config)
    genai.configure(api_key=GEMINI_API_KEY)
    llm_client = genai.Client()
    system_instruction_cache = create_gemini_system_prompt()
    print(f"✅ Gemini client initialized. Using model: {llm_model_name}")

def create_claude_system_prompt_cache() -> List[Dict]:
    """Create Claude system prompt with cache control for cost savings."""
    ksac_data = all_datasets.get('MNPS_KSACs', {})
    role_groups = all_datasets.get('MNPS_Role_Groups_by_KSAC_Similarity_FINAL', {})

    # Claude system prompt structure for caching
    system_blocks = [
        {
            "type": "text",
            "text": "You are an expert HR analyst and job classification specialist. Your task is to critique an AI's job role assignment against provided ground truth and comprehensive reference data (KSACs, Role Groups). Your analysis must focus on identifying plausible alternative roles and categorizing similarity/dissimilarity based on the provided reference information."
        },
        {
            "type": "text",
            "text": f"**MNPS KSAC Reference Data:**\n{json.dumps(ksac_data, indent=2)[:8000]}...",
            "cache_control": {"type": "ephemeral"}
        },
        {
            "type": "text",
            "text": f"**Role Group Similarity Clusters:**\n{json.dumps(role_groups, indent=2)[:8000]}...",
            "cache_control": {"type": "ephemeral"}
        }
    ]
    return system_blocks

def create_gemini_system_prompt() -> str:
    """Create a unified system instruction string for Gemini."""
    ksac_data = all_datasets.get('MNPS_KSACs', {})
    role_groups = all_datasets.get('MNPS_Role_Groups_by_KSAC_Similarity_FINAL', {})

    # Gemini uses a single system instruction string
    system_prompt = f"""You are an expert HR analyst and job classification specialist. Your task is to critique an AI's job role assignment against provided ground truth and comprehensive reference data (KSACs, Role Groups). Your analysis must focus on identifying plausible alternative roles and categorizing similarity/dissimilarity based on the provided reference information. Adhere strictly to the JSON output format provided in the user prompt.

**MNPS KSAC Reference Data:**
{json.dumps(ksac_data, indent=2)[:8000]}...

**Role Group Similarity Clusters:**
{json.dumps(role_groups, indent=2)[:8000]}..."""
    return system_prompt

# Execute the setup based on the chosen mode
if MODEL_MODE == 'LLAMA_LOCAL':
    setup_llama_local()
elif MODEL_MODE == 'CLAUDE_API':
    setup_claude_api()
elif MODEL_MODE == 'GEMINI_API':
    setup_gemini_api()



## 4. Role Analysis Engine

In [ ]:
# =========================================================================
# 🆕 Unified Role Analyzer Class
# =========================================================================

class RoleAnalyzer:
    def __init__(self, df: pd.DataFrame, datasets: Dict, model_mode: str, client, tokenizer=None, system_cache=None):
        self.df = df
        self.datasets = datasets
        self.model_mode = model_mode
        self.client = client
        self.tokenizer = tokenizer
        self.system_cache = system_cache
        self.llm_model_name = llm_model_name

    def create_prompt(self, row: pd.Series) -> str:
        """Create the dynamic prompt with analysis questions and output format."""
        # 1. Prepare Ground Truth
        job_title_original = row['job_title_original']
        ground_truth = next((
            item.get('major_role_group', 'N/A') 
            for item in self.datasets.get('Ground_Truth_Masterfile', []) 
            if item.get('Job Title') == job_title_original
        ), 'N/A')

        # 2. Prepare Similarity Data for context (though the LLM must use its cached/system view to answer Q3)
        role_groups = json.dumps(self.datasets.get('MNPS_Role_Groups_by_KSAC_Similarity_FINAL', []), indent=2)[:5000] # Truncate for safety
        ksacs = json.dumps(self.datasets.get('MNPS_KSACs', {}), indent=2)[:5000] # Truncate for safety

        prompt = f"""# JOB CLASSIFICATION ANALYSIS TASK

You are an expert HR analyst. Analyze the provided job information, the AI's classification, and the reference data (KSACs, Role Groups) to answer the questions below and return a structured JSON object. Adhere strictly to the JSON format.

## INPUT DATA
**A. Job Description (JD & Context):**
```
{row['full_job_description']}
```

**B. AI & Ground Truth:**
| Field | Value |
|:---|:---|
| **Job Code** | {row.get('Job Code', 'N/A')} |
| **Job Title (Original)** | {row['job_title_original']} |
| **AI Classified Role (major_role_group)** | {row.get('major_role_group', 'N/A')} |
| **Human Ground Truth Role** | {ground_truth} |

## ANALYSIS QUESTIONS (Answer in JSON Output)

1. **Plausibility Check**: Could the AI Classified Role have been another plausible job role based on the Job Description (A) and your general HR knowledge/KSAC data?
2. **Alternative Roles**: If plausible, list up to 5 specific alternative **MNPS Roles** (from the KSAC/Role Group data provided in your system instructions) that the JD content aligns with.
3. **Similarity Grouping**: From the Alternative Roles (Q2), which ones belong to the *same similarity group* as the AI Classified Role (B) based on the Role Group Similarity Clusters?
4. **Dissimilar Roles**: List up to 5 **MNPS Roles** (from the KSAC data) that are completely dissimilar to the AI Classified Role, demonstrating the boundaries of the classification.
5. **Analysis Reasoning**: Provide a 2-3 sentence summary of *why* the AI's classified role is either strong or weak, referencing JD sections (A) and Similarity Groups (Q3).

## REQUIRED OUTPUT FORMAT
Return **ONLY** a valid JSON object with this exact structure:
```json
{{
    "job_code": "{row.get('Job Code', 'N/A')}",
    "classified_role": "{row.get('major_role_group', 'N/A')}",
    "other_similar_roles": ["<Role1>", "<Role2>"],
    "other_dissimilar_roles": ["<Role3>", "<Role4>"],
    "plausible_alternatives_count": 0, // Integer count of the roles listed in 'other_similar_roles'
    "analysis_reasoning": "<Your 2-3 sentence summary.>"
}}
```
"""
        return prompt

    def generate_response(self, prompt: str) -> Dict:
        """Generate analysis using the selected LLM client/API."""
        messages = []
        response_text = ""

        if self.model_mode == 'CLAUDE_API':
            from anthropic import APIError, RateLimitError
            messages = [{"role": "user", "content": prompt}]
            try:
                response = call_claude_with_retry(
                    client=self.client,
                    model=self.llm_model_name,
                    messages=messages,
                    max_tokens=MAX_TOKENS,
                    temperature=TEMPERATURE,
                    system=self.system_cache if ENABLE_PROMPT_CACHING else None,
                    max_retries=MAX_RETRIES
                )
                response_text = response.content[0].text
            except Exception as e:
                return {"error": str(e), "status": "failed"}

        elif self.model_mode == 'GEMINI_API':
            from google.genai import types
            messages = [{'role': 'user', 'parts': [{'text': prompt}]}]
            try:
                response = self.client.models.generate_content(
                    model=self.llm_model_name,
                    contents=messages,
                    config=types.GenerateContentConfig(
                        system_instruction=self.system_cache,
                        temperature=TEMPERATURE,
                        max_output_tokens=MAX_TOKENS
                    )
                )
                response_text = response.text
            except Exception as e:
                 return {"error": str(e), "status": "failed"}

        elif self.model_mode == 'LLAMA_LOCAL':
            import torch
            messages = [
                {"role": "system", "content": self.system_cache},
                {"role": "user", "content": prompt}
            ]
            formatted_prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = self.tokenizer(
                formatted_prompt, return_tensors="pt", truncation=True, max_length=128000 # Max context size
            ).to(self.client.device)

            with torch.no_grad():
                outputs = self.client.generate(
                    **inputs,
                    max_new_tokens=MAX_TOKENS,
                    temperature=TEMPERATURE,
                    do_sample=True
                )
            response_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            # Clean LLAMA output: remove the user prompt/system prompt echo
            if "assistant" in response_text:
                response_text = response_text.split("assistant")[-1].strip()
            elif "user" in response_text:
                 response_text = response_text.split("user")[-1].strip()

        # Parse JSON response
        try:
            result_text = extract_json_from_response(response_text)
            result = json.loads(result_text)
            result['status'] = 'success'
        except Exception as e:
            result = {"error": f"JSON Parsing Failed: {e}. Raw: {response_text[:300]}...", "status": "failed"}

        # Add model metadata
        result['model_used'] = self.llm_model_name
        return result

    def process_all(self) -> pd.DataFrame:
        """Process all records in the DataFrame."""
        all_results = []
        print(f"\n🚀 Starting batch analysis using {self.model_mode} ({self.llm_model_name})...")

        for idx, row in tqdm(self.df.iterrows(), total=len(self.df), desc="Processing records"):
            prompt = self.create_prompt(row)
            result = self.generate_response(prompt)

            # Ensure essential input fields are present in the final result
            result_row = {
                'Job Code': row.get('Job Code', 'N/A'),
                'Classified Role': row.get('major_role_group', 'N/A'),
                'Model Used': result.get('model_used', self.llm_model_name),
                'Analysis Status': result.get('status', 'failed'),
                'Other Similar Roles': ", ".join(result.get('other_similar_roles', [])),
                'Other Dissimilar Roles': ", ".join(result.get('other_dissimilar_roles', [])),
                'Analysis Reasoning': result.get('analysis_reasoning', result.get('error', 'N/A')),
                'Plausible Alternatives Count': result.get('plausible_alternatives_count', 0)
            }
            all_results.append(result_row)

        df_results = pd.DataFrame(all_results)
        return df_results

# =========================================================================
# 🔨 Core Utility Functions (Moved from old notebooks)
# =========================================================================

def call_claude_with_retry(client, model, messages, max_tokens, temperature, system=None, max_retries=MAX_RETRIES):
    """Claude API call with exponential backoff retry logic."""
    from anthropic import APIError, RateLimitError, APITimeoutError
    for attempt in range(max_retries):
        try:
            params = {
                "model": model,
                "max_tokens": max_tokens,
                "temperature": temperature,
                "messages": messages
            }
            if system:
                params["system"] = system
            response = client.messages.create(**params)
            return response
        except RateLimitError:
            wait_time = RETRY_DELAY * (2 ** attempt)
            print(f"⏳ Rate limited. Waiting {wait_time:.1f}s... (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait_time)
        except APITimeoutError:
            wait_time = RETRY_DELAY * (2 ** attempt)
            print(f"⏳ Timeout. Retrying in {wait_time:.1f}s... (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait_time)
        except APIError as e:
            if attempt < max_retries - 1:
                wait_time = RETRY_DELAY
                print(f"⚠️ API error: {str(e)[:100]}")
                print(f"⏳ Retrying in {wait_time:.1f}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                raise
    raise Exception("LLM generation failed after all retries.")

def extract_json_from_response(text: str) -> str:
    """Extract JSON from LLM response, handling various markdown formats."""
    import re
    # 1. Look for ```json ... ``` block
    json_match = re.search(r'```json\s*([\s\S]*?)```', text)
    if json_match:
        return json_match.group(1).strip()
    # 2. Look for ``` ... ``` block
    json_match = re.search(r'```\s*([\s\S]*?)```', text)
    if json_match:
        return json_match.group(1).strip()
    # 3. Try to find the raw JSON object starting with '{' and ending with '}'
    obj_match = re.search(r'\{[\s\S]*\}', text)
    if obj_match:
        return obj_match.group(0).strip()
    # 4. If all fails, return raw text and let json.loads handle the error
    return text.strip()


## 5. Execution and Final Results

In [ ]:
analyzer = RoleAnalyzer(
    df=df_analysis,
    datasets=all_datasets,
    model_mode=MODEL_MODE,
    client=llm_client,
    tokenizer=llm_tokenizer,
    system_cache=system_instruction_cache
)

df_final_results = analyzer.process_all()
df_final_results['Plausible Alternatives Count'] = pd.to_numeric(df_final_results['Plausible Alternatives Count'], errors='coerce').fillna(0).astype(int)


## 6. Results Visualization and Export

In [ ]:
# =========================================================================
# 📊 Visualizations
# =========================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("\n📊 Generating visualizations...")
os.makedirs(CURRENT_RUN_PATH, exist_ok=True)

# --- 1. Plausible Alternatives Distribution (Histogram) ---
try:
    plt.figure(figsize=(10, 6))
    sns.histplot(df_final_results['Plausible Alternatives Count'], bins=range(0, 6), discrete=True, kde=False, color='#4ECDC4', edgecolor='black')
    plt.title('Distribution of Plausible Alternative Roles', fontweight='bold')
    plt.xlabel('Number of Plausible Alternatives Identified')
    plt.ylabel('Frequency (Classifications)')
    plt.xticks(range(0, 6))
    plt.grid(axis='y', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(CURRENT_RUN_PATH, 'alternatives_distribution.png'), dpi=300)
    plt.show()
    print("  ✅ Created Alternatives Distribution")
except Exception as e:
    print(f"  ❌ Error creating Alternatives Distribution: {e}")


# --- 2. Role Similarity Network (Top 10 Roles) ---
try:
    G = nx.Graph()
    df_top_roles = df_analysis['major_role_group'].value_counts().nlargest(10).index.tolist()

    # Add nodes and edges based on actual classifications and identified alternatives
    for _, row in df_final_results.iterrows():
        classified_role = row['Classified Role']
        similar_roles_list = [r.strip() for r in row['Other Similar Roles'].split(',') if r.strip()]

        if classified_role in df_top_roles:
            G.add_node(classified_role, node_type='Classified', size=20, color='red')
        else:
            G.add_node(classified_role, node_type='Classified', size=10, color='gray')

        for alt_role in similar_roles_list:
            if alt_role not in G:
                G.add_node(alt_role, node_type='Alternative', size=15, color='blue')
            
            # Add edge representing potential confusion
            G.add_edge(classified_role, alt_role, weight=1)

    # Create layout and plot (Interactive Plotly)
    pos = nx.spring_layout(G, k=0.5, iterations=50)
    fig = go.Figure(data=create_plotly_network_traces(G, pos), 
                    layout=go.Layout(
                        title='Role Confusion Network (Classified vs. Alternatives)',
                        showlegend=True,
                        hovermode='closest',
                        margin=dict(b=20, l=5, r=5, t=40),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        height=700
                    ))
    output_path_html = os.path.join(CURRENT_RUN_PATH, 'role_confusion_network.html')
    fig.write_html(output_path_html)
    print("  ✅ Created Interactive Role Similarity Network")
except Exception as e:
    print(f"  ❌ Error creating Role Similarity Network: {e}")


# --- 3. Analysis Status Pie Chart ---
try:
    plt.figure(figsize=(8, 8))
    status_counts = df_final_results['Analysis Status'].value_counts()
    colors = ['#4ECDC4', '#FF6B6B']
    plt.pie(status_counts, labels=status_counts.index, colors=colors[:len(status_counts)], autopct='%1.1f%%', startangle=90)
    plt.title('LLM Analysis Status', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(CURRENT_RUN_PATH, 'analysis_status_pie.png'), dpi=300)
    plt.show()
    print("  ✅ Created Analysis Status Pie Chart")
except Exception as e:
    print(f"  ❌ Error creating Analysis Status Pie Chart: {e}")


def create_plotly_network_traces(G, pos):
    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines'
    )

    node_x = []
    node_y = []
    node_text = []
    node_color = []
    node_size = []

    for node, attrs in G.nodes(data=True):
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(f"{node}<br>Type: {attrs.get('node_type')}<br>Links: {G.degree(node)}")
        node_color.append(attrs.get('color', 'grey'))
        node_size.append(attrs.get('size', 10))

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        hoverinfo='text',
        text=node_text,
        marker=dict(
            color=node_color,
            size=node_size,
            line_width=2
        )
    )

    return [edge_trace, node_trace]


## 7. Final Export and Cleanup

In [ ]:
# Save the final detailed results to CSV
results_csv_path = os.path.join(CURRENT_RUN_PATH, f'classification_analysis_{MODEL_MODE.lower()}.csv')
df_final_results.to_csv(results_csv_path, index=False)

print("\n========================================================================")
print("✅ Analysis Complete")
print(f"Results saved to: {results_csv_path}")
print(f"Total Success Rate: {(df_final_results['Analysis Status'] == 'success').mean():.1%}")
print("========================================================================")

# Display final results (excluding the raw JD for brevity)
display(df_final_results.drop(columns=['Model Used', 'Analysis Status']).head())

# Download the results directory in Colab (optional, useful for LLAMA_LOCAL)
try:
    import shutil
    zip_filename = 'analysis_results.zip'
    shutil.make_archive('analysis_results', 'zip', CURRENT_RUN_PATH)
    files.download(zip_filename)
    print(f"\n⬇️ Download initiated for {zip_filename}")
except Exception as e:
    print(f"⚠️ Download skipped (likely using local mode or drive not mounted). Files saved to {CURRENT_RUN_PATH}. Error: {e}")

## 8. How to Use Different Models

To switch models, simply change the `MODEL_MODE` variable in **Section 1 (Configuration)** and re-run all cells. 

| Model Mode | Required Setup | Recommended Environment | Notes |
|:---|:---|:---|:---|
| **CLAUDE_API** | `ANTHROPIC_API_KEY` (in Colab Secrets) | Standard/High RAM Runtime | Highly cost-efficient using API caching. |
| **LLAMA_LOCAL** | No key needed. | **A100/V100 GPU** (High VRAM) | Runs the model locally using 4-bit quantization. Slow download time for model weights. |
| **GEMINI_API** | `GEMINI_API_KEY` (in Colab Secrets) | Standard/High RAM Runtime | Uses Gemini's `system_instruction` for context, similar to Claude's caching for efficiency. |

### 🔑 Gemini API Setup Details

1.  **API Key**: Obtain a Gemini API key from the Google AI for Developers platform.
2.  **Colab Secret**: In your Colab notebook, click the 🔑 **Secrets** icon on the left sidebar.
3.  **Add Secret**: Create a new secret named `GEMINI_API_KEY` and paste your key as the value.
4.  **Model**: The notebook uses `gemini-2.5-pro` (configurable via `GEMINI_MODEL_NAME`) because of its strong reasoning and large output capacity, which is ideal for complex classification tasks.
5.  **System Instruction**: The reference data (KSACs, Role Groups) is packaged as a single `system_instruction` string, which is an effective way to guide the model's behavior and provide context over the entire run.

This setup ensures all models use a consistent prompt structure while leveraging their native features for performance and efficiency.
